Investigate how to find a better signal of how eddies change over time. 

### Tried already:
- Plot absolute quantities (pigments, chl, pfts) over eddy lifetime and relative pigment ratios (pigment / tchla). Absolute quantities show some divergence in target eddy trends but not that significant overall. When dividing by chlorophyll ratios there doesn't seem to be any difference over bins of normalized eddy lifetime for cyclones vs. anticyclones.

### Seems to Work (Tried This Week):
- Limit "target" eddies to only those that actually cross the GS. Previously eddies that meandered off on same side were also counted as long as within 150 km. However GS conditions are kind of unpredictable so this introduces random noise.

- Remove eddy-day composites that had unreasonable results after taking aggregate. Some large error bars were due to pigment / tchla ratios that didn't make logical sense. For example a pigment like DVChla is a component of the sum to TChla so it can't possibly exceed, but SDP predicted a ratio > 1. Or if > half of pixels had 0 Tchla.

- Use crossing the Gulf Stream phase (partition into 2 phases) instead of letting the lifetime bins overall sit on the x-axis. That is, on the x-axis, we have two partitions. Left portion of x-axis is normalized 0-1 duration of when the eddy was still on its "original" side of the Gulf Stream. That is, either on its original side or within 150 km of the Gulf Stream (may test sensitivity or revise this 150 number). Then, we look for an eddy's first crossing of the Gulf Stream like truly into a new region detached from the Gulf Stream we're confident that it's meandered off already. Have those snapshots be normalized 0-1. See if that visual allows for improved distinction, as eddies may cross the GS at various times in their normalized lifetime (some 0.2, some 0.8, so it doesn't line up well).
    - One concern is an eddy that loops back and gets close to the GS again. Initially filter these out and don't consider them. If that cuts down observations by too much keep them in and just consider end of first phase as first departure of original region.

- For the case study focus on only the strongest eddies (of the remaining after above conditions)

### Ideas / Currently Implementing:
- Determine a way to correct for the season. I.e. if seasonal trends coincide with/against certain times that we're tracking the eddies what impact will that have? Will it amplify or dampen any trends we're seeing? Technically over a large enough sample we can be confident but if time of year has some correlation with cloud cover and what we see/track then that will mess things up.

- 

In [ ]:
from pathlib import Path
from typing import cast
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from matplotlib.axes import Axes
from matplotlib.figure import Figure
from matplotlib.ticker import MaxNLocator

PROJECT_ROOT = Path('/Users/jerry/school/research/eddy-tracking')
sys.path.insert(0, str(PROJECT_ROOT))
from eddy_tracking.utils.analysis_period import restrict_to_analysis_period
from eddy_tracking.config import load_analysis_config, read_settings, resolve_gold_dir, resolve_output_dir
from eddy_tracking.preprocess.streamline import compute_signed_distance_km, index_centerlines_by_date
from eddy_tracking.preprocess.tracks import load_track_observations
from eddy_tracking.utils.plankton_layout import COPERNICUS_LABELS, SIZE_CLASS_CELLS, SIZE_CLASS_FIELDS, SIZE_CLASS_GRID_SHAPE, find_spare_cells
from eddy_tracking.validation.quality import mask_invalid_plankton_groups, keep_valid_sdp_rows

EXPERIMENT = 'gulf_stream_20240305_20260531'
N_PHASE_BINS = 2
N_BOOTSTRAP = 2000
RANDOM_SEED = 2026
NEAR_AXIS_KM = 150
DATA_DIR = PROJECT_ROOT / 'data' / EXPERIMENT
cfg = load_analysis_config(EXPERIMENT)
polarity_names = ('cyclone', 'anticyclone')
target_classes = {'cyclone': 'NS', 'anticyclone': 'SN'}
polarity_colors = {'cyclone': '#2166ac', 'anticyclone': '#b2182b'}
polarity_labels = {'cyclone': 'Target cyclones', 'anticyclone': 'Target anticyclones'}
polarity_sides = {'cyclone': ('north side', 'south side'), 'anticyclone': ('south side', 'north side')}
identity_columns = ['polarity', 'track_id']
groups = SIZE_CLASS_FIELDS
group_labels = COPERNICUS_LABELS
pigments = ['Tchla', 'Fuco', 'HexFuco', 'ButFuco', 'Perid', 'Chlc12', 'Chlc3', 'Allo', 'MV_chlb', 'Neo', 'Viola', 'Zea', 'DV_chla']
pigment_labels = {
    'Tchla': 'Total chlorophyll-a', 'Fuco': 'Fucoxanthin', 'HexFuco': "19'-Hex-fucoxanthin", 'ButFuco': "19'-But-fucoxanthin",
    'Perid': 'Peridinin', 'Chlc12': 'Chlorophyll-c1+c2', 'Chlc3': 'Chlorophyll-c3', 'Allo': 'Alloxanthin',
    'MV_chlb': 'Monovinyl chlorophyll-b', 'Neo': 'Neoxanthin', 'Viola': 'Violaxanthin',
    'Zea': 'Zeaxanthin', 'DV_chla': 'Divinyl chlorophyll-a',
}
variable_labels = {'CHL': 'CHL', **group_labels, **pigment_labels}
panel_letters = 'abcdefghijklmnop'
phase_ticks = ([0, 0.5, 1, 1.5, 2], ['0', '0.5', '1 | 0', '0.5', '1'])

eddy_tracks = restrict_to_analysis_period(pd.read_parquet(DATA_DIR / 'silver/gulf_stream/eddy_movement.parquet'), cfg)
plankton = restrict_to_analysis_period(pd.read_parquet(DATA_DIR / 'gold/eddy_plankton_table.parquet'), cfg)
pigment_table = keep_valid_sdp_rows(restrict_to_analysis_period(pd.read_parquet(DATA_DIR / 'gold/eddy_pigment_table.parquet'), cfg))
pigment_table['polarity'] = cast(pd.Series, pigment_table['polarity']).map({0: 'anticyclone', 1: 'cyclone'})
pigment_table = pigment_table.rename(columns={f'eddy_mean_{pigment}': pigment for pigment in pigments})
# Only the eddies first detected on one side of the axis and last detected on the other: cyclones from north to south, anticyclones from south to north
target_tracks = eddy_tracks.loc[eddy_tracks['movement'].eq(cast(pd.Series, eddy_tracks['polarity']).map(target_classes)), identity_columns + ['birth_date', 'death_date']]

observations = load_track_observations(resolve_output_dir(EXPERIMENT, 'eddy_track')).drop(columns=['contour_lon', 'contour_lat']).merge(target_tracks[identity_columns], on=identity_columns).sort_values(identity_columns + ['date'], ignore_index=True)
centerlines = index_centerlines_by_date(pd.read_parquet(DATA_DIR / 'silver/gulf_stream/streamline.parquet'))
# Distance from the daily axis toward the side a target eddy ends on, so positive is south of the axis for a cyclone and north of it for an anticyclone
observations['across_km'] = cast(pd.Series, observations['polarity']).map({'cyclone': -1, 'anticyclone': 1}) * [
    compute_signed_distance_km(centerlines[day.date()].lon, centerlines[day.date()].lat, lon, lat)[0]
    for day, lon, lat in zip(observations['date'], observations['center_lon'], observations['center_lat'])
]


def add_phase_position(table: pd.DataFrame) -> pd.DataFrame:
    # The departure is the first day an eddy is farther than NEAR_AXIS_KM past the axis; a target eddy that never gets that far has no second phase and drops out here
    departure = cast(pd.Series, observations.loc[observations['across_km'].gt(NEAR_AXIS_KM)].groupby(identity_columns)['date'].min()).rename('departure_date').reset_index()
    table = table.merge(target_tracks, on=identity_columns, suffixes=('', '_track')).merge(departure, on=identity_columns)
    before = table['date'] < table['departure_date']
    start = table['birth_date'].where(before, table['departure_date'])
    end = table['departure_date'].where(before, table['death_date'])
    # 0 to 1 is the time on the original side, from the first detection to the departure, and 1 to 2 the time on the new side, up to the last detection; a composite midpoint can fall outside the track, hence the clip
    table['phase_position'] = ((table['date'] - start).dt.days / (end - start).dt.days.clip(lower=1)).clip(0, 1) + ~before
    return table


track_days = add_phase_position(observations)
target_plankton = add_phase_position(cast(pd.DataFrame, plankton[identity_columns + ['date', 'n_pixels', 'CHL'] + groups + [f'{group}_n_pixels' for group in groups]]))
target_pigments = add_phase_position(cast(pd.DataFrame, pigment_table[identity_columns + ['date'] + pigments]))
plankton_settings = read_settings(resolve_gold_dir(EXPERIMENT, 'eddy_plankton_table.parquet'), 'collocate-plankton')
target_plankton = mask_invalid_plankton_groups(target_plankton, plankton_settings)
key_columns = identity_columns + ['date', 'phase_position']
analysis = pd.concat([
    target_plankton[key_columns].assign(source='chl', variable='CHL', concentration=target_plankton['CHL']),
    target_plankton[key_columns + groups].melt(id_vars=key_columns, var_name='variable', value_name='concentration').assign(source='plankton'),
    # Each SDP pigment is its interior mean divided by the interior mean of Tchla, as in Figure 3 of figures.ipynb
    cast(pd.DataFrame, target_pigments[key_columns]).join(cast(pd.DataFrame, target_pigments[pigments[1:]]).div(target_pigments['Tchla'], axis=0)).melt(id_vars=key_columns, var_name='variable', value_name='concentration').assign(source='sdp'),
], ignore_index=True).dropna(subset=['concentration']).sort_values(['source', 'variable'] + identity_columns + ['date'], ignore_index=True)

plt.rcParams.update({
    'font.family': 'sans-serif', 'font.sans-serif': ['Arial', 'Helvetica', 'DejaVu Sans'],
    'mathtext.fontset': 'custom', 'mathtext.rm': 'Arial', 'mathtext.it': 'Arial:italic', 'mathtext.bf': 'Arial:bold',
    # matplotlib 3.11 warns about its default calligraphic face, Apple Chancery, even though no label uses it
    'mathtext.cal': 'Arial',
    'font.size': 8, 'axes.titlesize': 8, 'axes.labelsize': 8, 'xtick.labelsize': 8, 'ytick.labelsize': 8, 'legend.fontsize': 8,
    'axes.linewidth': 0.6, 'xtick.major.width': 0.6, 'ytick.major.width': 0.6, 'xtick.major.size': 2.5, 'ytick.major.size': 2.5,
    'axes.spines.top': False, 'axes.spines.right': False, 'legend.frameon': False,
    'figure.dpi': 150, 'savefig.dpi': 300,
})

after = track_days['phase_position'].ge(1)
track_phases = cast(pd.DataFrame, track_days.assign(
    before=~after, after=after,
    returned=after & track_days['across_km'].le(NEAR_AXIS_KM), crossed_back=after & track_days['across_km'].lt(0),
).groupby(identity_columns).agg(
    days_before=('before', 'sum'), days_after=('after', 'sum'), returned=('returned', 'any'), crossed_back=('crossed_back', 'any'),
)).reset_index()
display(pd.DataFrame({
    'target eddies': target_tracks.groupby('polarity').size(),
    f'reach {NEAR_AXIS_KM} km past the axis': track_phases.groupby('polarity').size(),
    f'never back within {NEAR_AXIS_KM} km': (~track_phases['returned']).groupby(track_phases['polarity']).sum(),
    'never back across the axis': (~track_phases['crossed_back']).groupby(track_phases['polarity']).sum(),
    'median days on the original side': track_phases.groupby('polarity')['days_before'].median(),
    'median days on the new side': track_phases.groupby('polarity')['days_after'].median(),
}))
display(cast(pd.DataFrame, analysis.drop_duplicates(['source'] + key_columns).assign(phase=lambda rows: np.where(rows['phase_position'].lt(1), 'original side', 'new side')).groupby(['source', 'polarity', 'phase']).agg(
    eddies=('track_id', 'nunique'), composites=('date', 'size'),
).unstack('phase')).reindex(columns=['original side', 'new side'], level='phase'))

In [ ]:
bin_centers = (np.arange(2 * N_PHASE_BINS) + 0.5) / N_PHASE_BINS


def summarize_phases(table: pd.DataFrame, metrics: tuple[str, ...]) -> pd.DataFrame:
    table = table.assign(phase_bin=np.minimum((table['phase_position'] * N_PHASE_BINS).astype(int), 2 * N_PHASE_BINS - 1))
    table['initial'] = table.groupby(['source', 'variable'] + identity_columns)['concentration'].transform('first')
    table['change'] = table['concentration'] - table['initial']
    eddy_bins = cast(pd.DataFrame, table.groupby(['source', 'variable'] + identity_columns + ['phase_bin']).agg(
        concentration=('concentration', 'mean'), change=('change', 'mean'), initial=('initial', 'first'),
    )).reset_index()
    rng = np.random.default_rng(RANDOM_SEED)
    summary_rows = []
    for keys, source_bins in eddy_bins.groupby(['source', 'polarity']):
        source, polarity = cast(tuple[str, str], keys)
        eddy_ids = sorted(source_bins['track_id'].unique())
        draws = rng.integers(0, len(eddy_ids), size=(N_BOOTSTRAP, len(eddy_ids)))
        for variable, variable_bins in source_bins.groupby('variable'):
            initial = variable_bins.groupby('track_id')['initial'].first().reindex(eddy_ids).to_numpy(dtype=float)
            for metric in metrics:
                matrix = variable_bins.pivot(index='track_id', columns='phase_bin', values='change' if metric == 'percent' else metric).reindex(index=eddy_ids, columns=range(2 * N_PHASE_BINS)).to_numpy(dtype=float)
                present = np.isfinite(matrix)
                counts = present.sum(axis=0)
                # The percent change of a bin is the mean change over the eddies in it divided by the mean first observation of those same eddies; a mean of per-eddy percents blows up when one first observation is near zero
                denominator = np.where(present, initial[:, None], 0).sum(axis=0) / 100 if metric == 'percent' else counts
                means = np.divide(np.nansum(matrix, axis=0), denominator, out=np.full(2 * N_PHASE_BINS, np.nan), where=counts > 0)
                sampled = matrix[draws]  # (n_eddies, n_bins) -> (n_bootstrap, n_eddies, n_bins)
                sampled_present = np.isfinite(sampled)
                sampled_counts = sampled_present.sum(axis=1)
                sampled_denominator = np.where(sampled_present, initial[draws][:, :, None], 0).sum(axis=1) / 100 if metric == 'percent' else sampled_counts
                sampled_means = np.divide(np.nansum(sampled, axis=1), sampled_denominator, out=np.full((N_BOOTSTRAP, 2 * N_PHASE_BINS), np.nan), where=sampled_counts > 0)
                low, high = np.nanquantile(sampled_means, [0.025, 0.975], axis=0)
                summary_rows.append(pd.DataFrame({
                    'source': source, 'polarity': polarity, 'variable': variable, 'metric': metric, 'phase_bin': range(2 * N_PHASE_BINS), 'phase_position': bin_centers,
                    'mean': means, 'ci_low': np.where(counts >= 3, low, np.nan), 'ci_high': np.where(counts >= 3, high, np.nan), 'n_eddies': counts, 'n_tracks': len(eddy_ids),
                }))
    return pd.concat(summary_rows, ignore_index=True)


phase_summary = summarize_phases(analysis, ('concentration', 'percent'))


def draw_phases(ax: Axes, source: str, variable: str, metric: str) -> None:
    ax.axvline(1, color='#999999', linewidth=0.6, zorder=1)
    if metric != 'concentration':
        ax.axhline(0, color='#999999', linewidth=0.6, zorder=1)
    for polarity in polarity_names:
        color = polarity_colors[polarity]
        result = phase_summary.loc[
            phase_summary['source'].eq(source) & phase_summary['variable'].eq(variable)
            & phase_summary['polarity'].eq(polarity) & phase_summary['metric'].eq(metric)
        ].sort_values('phase_bin')
        intervals = result['ci_low'].notna()
        ax.errorbar(
            result.loc[intervals, 'phase_position'], result.loc[intervals, 'mean'],
            yerr=[result.loc[intervals, 'mean'] - result.loc[intervals, 'ci_low'], result.loc[intervals, 'ci_high'] - result.loc[intervals, 'mean']],
            fmt='none', ecolor=color, capsize=1.5, elinewidth=0.7, capthick=0.7, zorder=3,
        )
        ax.plot(result['phase_position'], result['mean'], marker='o', linestyle='-', color=color, linewidth=1.2, markersize=3.2, markeredgecolor='white', markeredgewidth=0.5, label=f'{polarity_labels[polarity]} (n = {result["n_tracks"].iloc[0]})', zorder=4)
    ax.xaxis.set_ticks(*phase_ticks)
    ax.set_xlim(0, 2)
    ax.grid(axis='y', color='#e5e5e5', linewidth=0.5)
    ax.set_axisbelow(True)
    locator = MaxNLocator(nbins=5, steps=[1, 2, 2.5, 5, 10])
    ax.yaxis.set_major_locator(locator)
    ticks = cast(np.ndarray, locator.tick_values(*ax.get_ylim()))
    if metric == 'concentration' and ax.dataLim.y0 >= 0:
        # The autoscale margin can dip below zero, which a concentration or a ratio never does
        ticks = ticks[ticks >= 0]
    ax.yaxis.set_ticks(ticks.tolist())
    ax.set_ylim(ticks[0], ticks[-1])


def label_sides(ax: Axes) -> None:
    # The first half of the axis is the north side for a cyclone and the south side for an anticyclone, so each half carries one word per polarity in that polarity's color
    for row, polarity in enumerate(polarity_names):
        for position, side in zip((0.25, 0.75), polarity_sides[polarity]):
            ax.annotate(side, (position, 0), xycoords='axes fraction', xytext=(0, -15 - 10 * row), textcoords='offset points', ha='center', va='top', color=polarity_colors[polarity])
    ax.set_xlabel('Fraction of time on each side\nof the Gulf Stream', labelpad=24)


def draw_grids(source: str, cells: list[tuple[int, int, str]], shape: tuple[int, int], figsize: tuple[float, float], value_label: str, percent_label: str) -> None:
    # cells are (row, column, variable); panels in the bottom row carry the side labels, and the first empty position holds the legend
    spare = find_spare_cells(cells, shape)
    for metric, ylabel in (('concentration', value_label), ('percent', percent_label)):
        grid_fig, grid_axes = cast(tuple[Figure, np.ndarray], plt.subplots(*shape, figsize=figsize, layout='constrained', squeeze=False))
        for letter, (row, column, variable) in zip(panel_letters, cells):
            ax = cast(Axes, grid_axes[row, column])
            draw_phases(ax, source, variable, metric)
            ax.set_title(rf'$\bf{{({letter})}}$ {variable_labels[variable]}', loc='left')
            if row == shape[0] - 1:
                label_sides(ax)
        for row, column in spare[1:]:
            cast(Axes, grid_axes[row, column]).remove()
        handles, labels = cast(Axes, grid_axes[cells[0][0], cells[0][1]]).get_legend_handles_labels()
        if spare:
            legend_ax = cast(Axes, grid_axes[spare[0]])
            legend_ax.axis('off')
            legend_ax.legend(handles, labels, loc='center', handlelength=1.6)
        else:
            grid_fig.legend(handles, labels, loc='outside lower center', ncol=2)
        grid_fig.supylabel(ylabel, fontsize=8)
        plt.show()


display(phase_summary.loc[phase_summary['metric'].eq('concentration')].groupby(['source', 'polarity', 'phase_bin'])['n_eddies'].min().unstack('phase_bin'))

## Copernicus CHL

In [ ]:
chl_fig, chl_axes = cast(tuple[Figure, np.ndarray], plt.subplots(1, 2, figsize=(6.69, 3.3), layout='constrained'))
for ax, letter, metric, title, ylabel in zip(cast(list[Axes], chl_axes.tolist()), panel_letters, ('concentration', 'percent'), ('Interior CHL', 'Change from first observation'), ('CHL (mg m$^{-3}$)', 'Change (%)')):
    draw_phases(ax, 'chl', 'CHL', metric)
    ax.set_title(rf'$\bf{{({letter})}}$ {title}', loc='left')
    ax.set_ylabel(ylabel)
    label_sides(ax)
chl_fig.legend(*cast(Axes, chl_axes[0]).get_legend_handles_labels(), loc='outside lower center', ncol=2)
plt.show()

## Copernicus plankton groups

In [ ]:
draw_grids('plankton', SIZE_CLASS_CELLS, SIZE_CLASS_GRID_SHAPE, (6.69, 7.6), 'Group chlorophyll-a (mg m$^{-3}$)', 'Change from first observation (%)')

## SDP pigments / Tchla

In [ ]:
draw_grids('sdp', [(index // 3, index % 3, variable) for index, variable in enumerate(pigments[1:])], (4, 3), (6.69, 7.4), 'Pigment / Tchla', 'Change in pigment / Tchla from first observation (%)')

In [ ]:
from matplotlib.lines import Line2D
from eddy_tracking.utils.pigment_clusters import resolve_pigment_clusters

# Okabe and Ito colors, which stay distinct under every color vision deficiency; blue and vermillion are left out because they mark the polarities
PFT_BLOCK_COLORS = ('#E69F00', '#009E73', '#CC79A7', '#56B4E9', '#000000')
N_PIGMENT_COLUMNS = 2
# Inches: the height of one row of panels with its gap, the margins above and below the panel grid, and the left edge of the block labels
PIGMENT_ROW_HEIGHT = 0.99
PIGMENT_MARGIN = 0.525
# Extra room under the panel grid for the side names and the x label under the lowest panels
SIDE_LABEL_MARGIN = 0.55
BLOCK_LABEL_X = 0.415
# One block per labeled cluster of the ratio tree in pace_sdp.ipynb, named after the PFTs whose diagnostic pigments it holds, with the cluster's pigments that occur in at least one of them
resolved_pigments = tuple(
    (cluster.label, PFT_BLOCK_COLORS[block % len(PFT_BLOCK_COLORS)], cluster.carrier_pigments)
    for block, cluster in enumerate(cluster for cluster in resolve_pigment_clusters(DATA_DIR, cfg) if cluster.labels)
)
print('\n'.join(f'{pft_label}: {", ".join(pft_pigments)}' for pft_label, _, pft_pigments in resolved_pigments))
block_rows = [-(-len(pft_pigments) // N_PIGMENT_COLUMNS) for *_, pft_pigments in resolved_pigments]
n_pigment_rows = sum(block_rows)
figure_width = 6.69
figure_height = 2 * PIGMENT_MARGIN + SIDE_LABEL_MARGIN + n_pigment_rows * PIGMENT_ROW_HEIGHT
fig = cast(Figure, plt.figure(figsize=(figure_width, figure_height)))
# A block with several PFTs lists one per line; the widest label sets where the brackets and the panels start
block_labels = [fig.text(BLOCK_LABEL_X / figure_width, 0.5, pft_label.replace(', ', ',\n'), color=color, va='center', ha='left') for pft_label, color, _ in resolved_pigments]
fig.draw_without_rendering()
bracket_x = (BLOCK_LABEL_X + max(text.get_window_extent().width for text in block_labels) / fig.dpi + 0.08) / figure_width
pigment_grid = fig.add_gridspec(
    n_pigment_rows, N_PIGMENT_COLUMNS, left=bracket_x + 0.64 / figure_width, right=0.985,
    bottom=(PIGMENT_MARGIN + SIDE_LABEL_MARGIN) / figure_height, top=1 - PIGMENT_MARGIN / figure_height, wspace=0.32, hspace=0.65,
)
pigment_axes = cast(np.ndarray, pigment_grid.subplots(sharex=True, squeeze=False))
bottom_axes: dict[int, Axes] = {}
row = 0
panel = 0
for (pft_label, color, pft_pigments), pft_rows, label_text in zip(resolved_pigments, block_rows, block_labels):
    for offset, pigment in enumerate(pft_pigments):
        ax = cast(Axes, pigment_axes[row + offset // N_PIGMENT_COLUMNS, offset % N_PIGMENT_COLUMNS])
        draw_phases(ax, 'sdp', pigment, 'concentration')
        ax.set_title(f'$\\bf{{({panel_letters[panel]})}}$ {pigment_labels[pigment]} / Tchla', loc='left')
        ax.tick_params(labelbottom=False)
        bottom_axes[offset % N_PIGMENT_COLUMNS] = ax
        panel += 1
    # The cells after the block's last pigment stay empty
    for column in range(len(pft_pigments) - (pft_rows - 1) * N_PIGMENT_COLUMNS, N_PIGMENT_COLUMNS):
        cast(Axes, pigment_axes[row + pft_rows - 1, column]).remove()
    pft_box = pigment_grid[row:row + pft_rows, :].get_position(fig)
    tick = 0.054 / figure_width
    fig.add_artist(Line2D([bracket_x + tick, bracket_x, bracket_x, bracket_x + tick], [pft_box.y0, pft_box.y0, pft_box.y1, pft_box.y1], transform=fig.transFigure, color=color, linewidth=1))  # pyright: ignore[reportAttributeAccessIssue]
    label_text.set_y((pft_box.y0 + pft_box.y1) / 2)
    row += pft_rows
# The lowest panel of each column carries the x tick labels
for ax in bottom_axes.values():
    ax.tick_params(labelbottom=True)
    label_sides(ax)
pigment_box = pigment_grid[:, :].get_position(fig)
fig.legend(*cast(Axes, pigment_axes[0, 0]).get_legend_handles_labels(), loc='upper center', bbox_to_anchor=((pigment_box.x0 + pigment_box.x1) / 2, 1 - 0.035 / figure_height), ncol=2, handlelength=1.8, columnspacing=1)
fig.text(0.054 / figure_width, (pigment_box.y0 + pigment_box.y1) / 2, 'Pigment / Tchla', rotation=90, va='center', ha='left')
plt.show()

Each block is one cluster of the pigment / Tchla tree over the target eddy-composites (`pace_sdp.ipynb`, cut at `CLUSTER_CUT_HEIGHT`). The label at left names every PFT whose diagnostic pigment sits in that cluster (Fuco for diatoms, Perid for dinoflagellates, ButFuco for pelagophytes, HexFuco for haptophytes, Allo for cryptophytes, MVchlb and Neo for green algae, Zea and DVchla for cyanobacteria; Kramer and Siegel, 2019), so a block names several PFTs when their markers share one cluster. A block shows the cluster's pigments that occur in at least one of its PFTs. Shared pigments such as Chlc12, Chlc3, and Viola do not name a block, and a cluster with no diagnostic pigment (Chlc3 alone at this cut) is left out. Chrysophytes have no diagnostic pigment among the SDP pigments and cannot be named.

Goal: Show that the interior CHL, pigments, and/or PFT classes change over an eddy's lifetime depending on where it moves